# LlmExpl Context Lab

Voir le contexte transformer les représentations

## Expérience 1 — Un même mot, deux contextes

Nous allons suivre **« avocat »** à travers les couches du modèle :

- ⚖️ « L'avocat plaide devant le tribunal. »
- 🥑 « Elle ajoute de l'avocat dans la salade. »

**Question : à quel moment les deux représentations commencent-elles à se distinguer ?**

### 1. Préparer le laboratoire

Nous utilisons le même modèle multilingue MPNet que dans LlmExpl et observons ses états internes *avant* le pooling.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from sklearn.decomposition import PCA
from torch.nn.functional import cosine_similarity
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TARGET_WORD = "avocat"
sentences = {
    "⚖️ Justice": "L'avocat plaide devant le tribunal.",
    "🥑 Cuisine": "Elle ajoute de l'avocat dans la salade.",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Calcul sur : {device}")

### 2. Charger le modèle et examiner la tokenisation

Les positions de caractères permettent de retrouver « avocat », qu'il corresponde à un ou plusieurs tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

print(f"Architecture : {model.config.model_type}")
print(f"Couches : {model.config.num_hidden_layers}")
print(f"Dimensions : {model.config.hidden_size}")

In [ ]:
def prepare(sentence, word):
    encoded = tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    start = sentence.lower().index(word.lower())
    end = start + len(word)
    indices = [i for i, (a, b) in enumerate(offsets) if b > start and a < end]
    if not indices:
        raise ValueError(f"Le mot {word!r} n'a pas été retrouvé.")
    return encoded, tokens, offsets, indices

prepared, rows = {}, []
for context, sentence in sentences.items():
    encoded, tokens, offsets, indices = prepare(sentence, TARGET_WORD)
    prepared[context] = (encoded, indices)
    for i, (token, offset) in enumerate(zip(tokens, offsets)):
        rows.append({"Contexte": context, "Position": i, "Token": token,
                     "Caractères": str(tuple(offset)),
                     "Mot suivi": "← avocat" if i in indices else ""})

pd.DataFrame(rows)

### 3. Suivre « avocat » à travers les couches

L'état 0 précède les couches Transformer. Les états 1 à 12 sont leurs sorties. Si le mot occupe plusieurs tokens, leurs vecteurs sont moyennés.

In [ ]:
trajectories = {}
with torch.inference_mode():
    for context, (encoded, indices) in prepared.items():
        inputs = {name: value.to(device) for name, value in encoded.items()}
        output = model(**inputs, output_hidden_states=True, return_dict=True)
        trajectories[context] = torch.stack([
            state[0, indices, :].mean(dim=0).cpu()
            for state in output.hidden_states
        ])

print(f"{len(next(iter(trajectories.values())))} états observés.")

### 4. Mesurer la séparation

Une similarité cosinus proche de 1 indique deux directions proches. Sa diminution révèle l'effet différenciateur du contexte.

In [ ]:
justice = trajectories["⚖️ Justice"]
cuisine = trajectories["🥑 Cuisine"]
similarities = cosine_similarity(justice, cuisine, dim=1).numpy()

metrics = pd.DataFrame({
    "État": np.arange(len(similarities)),
    "Étape": ["Représentation initiale"] + [f"Couche {i}" for i in range(1, len(similarities))],
    "Similarité cosinus": similarities,
    "Distance euclidienne": torch.linalg.vector_norm(justice - cuisine, dim=1).numpy(),
})
display(metrics.round(4))

fig = px.line(metrics, x="État", y="Similarité cosinus", markers=True,
              hover_name="Étape",
              title="Quand les deux « avocat » commencent-ils à diverger ?")
fig.update_traces(line=dict(width=3), marker=dict(size=9, color="#7c3aed"))
fig.update_layout(template="plotly_white", yaxis_range=[-0.05, 1.05],
                  hovermode="x unified")
fig.show()

### 5. Voir les trajectoires

La PCA projette les 26 représentations dans un même plan. C'est un trou de serrure utile, pas une image complète des 768 dimensions.

In [ ]:
contexts = list(trajectories)
vectors = np.vstack([trajectories[c].numpy() for c in contexts])
pca = PCA(n_components=2)
xy = pca.fit_transform(vectors)

rows, cursor = [], 0
for context in contexts:
    for state in range(len(trajectories[context])):
        rows.append({"Contexte": context, "État": state,
                     "Étape": "Représentation initiale" if state == 0 else f"Couche {state}",
                     "PCA 1": xy[cursor, 0], "PCA 2": xy[cursor, 1]})
        cursor += 1
trajectory_df = pd.DataFrame(rows)

fig = px.line(trajectory_df, x="PCA 1", y="PCA 2", color="Contexte",
              markers=True, text="État", hover_name="Étape",
              color_discrete_map={"⚖️ Justice": "#2563eb", "🥑 Cuisine": "#16a34a"},
              title="Trajectoire d'« avocat » à travers les couches")
fig.update_traces(line=dict(width=3), marker=dict(size=9), textposition="top center")
variance = 100 * pca.explained_variance_ratio_.sum()
fig.update_layout(template="plotly_white",
                  title=f"Trajectoire d'« avocat » à travers les couches<br><sup>Projection : {variance:.1f} % de variance conservée</sup>")
fig.show()

### 6. Lire l'expérience

Après exécution, observons :

1. la tokenisation d'« avocat » ;
2. les couches où la similarité diminue le plus ;
3. la forme des trajectoires dans la projection ;
4. ce que cette image permet — ou non — d'affirmer.

> **Prudence :** une séparation montre un effet du contexte. Elle ne prouve pas à elle seule que le modèle possède deux concepts humains parfaitement distincts.

## Expérience 2 — Deux sens de `mouse`, deux instruments

Nous comparons maintenant le même mot cible dans deux contextes fortement polysémiques :

- 🐭 `The mouse ate the cheese.`
- 🖱️ `I clicked the icon with the mouse.`

La vue 3D montre la géométrie projetée ; la courbe mesure directement la proximité dans l'espace complet.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mouse_sentences = {
    "🐭 Animal": "The mouse ate the cheese.",
    "🖱️ Computer": "I clicked the icon with the mouse.",
}

mouse_trajectories = {}
with torch.inference_mode():
    for context, sentence in mouse_sentences.items():
        encoded, _, _, indices = prepare(sentence, "mouse")
        inputs = {name: value.to(device) for name, value in encoded.items()}
        output = model(**inputs, output_hidden_states=True, return_dict=True)
        mouse_trajectories[context] = torch.stack([
            state[0, indices, :].mean(dim=0).cpu()
            for state in output.hidden_states
        ])

contexts = list(mouse_trajectories)
vectors = np.vstack([mouse_trajectories[c].numpy() for c in contexts])
pca3 = PCA(n_components=3)
xyz = pca3.fit_transform(vectors)

n_states = len(mouse_trajectories[contexts[0]])
layers = np.arange(n_states)
labels = np.array(["Représentation initiale"] + [f"Couche {i}" for i in layers[1:]])
similarities = cosine_similarity(
    mouse_trajectories[contexts[0]], mouse_trajectories[contexts[1]], dim=1
).numpy()

fig = make_subplots(
    rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
    column_widths=[0.62, 0.38], horizontal_spacing=0.05,
    subplot_titles=("Trajectoires PCA 3D", "Similarité cosinus"),
)
colors = {"🐭 Animal": "#2563eb", "🖱️ Computer": "#f97316"}

cursor = 0
for context in contexts:
    points = xyz[cursor:cursor + n_states]
    fig.add_trace(go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode="lines+markers", name=context, customdata=labels,
        line=dict(color=colors[context], width=6),
        marker=dict(color=colors[context], size=5),
        hovertemplate=("<b>%{customdata}</b><br>" + context +
                       "<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}<extra></extra>"),
    ), row=1, col=1)
    cursor += n_states

fig.add_trace(go.Scatter(
    x=layers, y=similarities, mode="lines+markers", name="Similarité",
    customdata=labels, line=dict(color="#7c3aed", width=3),
    marker=dict(size=8),
    hovertemplate="<b>%{customdata}</b><br>Similarité=%{y:.4f}<extra></extra>",
), row=1, col=2)

variance = 100 * pca3.explained_variance_ratio_
fig.update_scenes(
    xaxis_title=f"PC1 ({variance[0]:.1f} %)",
    yaxis_title=f"PC2 ({variance[1]:.1f} %)",
    zaxis_title=f"PC3 ({variance[2]:.1f} %)",
)
fig.update_xaxes(title_text="État / couche", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="Similarité cosinus", range=[-0.05, 1.05], row=1, col=2)
fig.update_layout(
    template="plotly_white", height=650, width=1200,
    title="Un même token, deux contextes : ce que l'on voit et ce que l'on mesure",
    legend=dict(orientation="h", y=1.08, x=0),
)
fig.show()

## Expérience 3 — Isoler le déplacement produit par le contexte

La représentation brute contient une forte composante commune liée au token `mouse`. Nous la retirons, couche par couche, en prenant `mouse` isolé comme point de référence :

$$\Delta_l^{contexte} = h_l^{contexte} - h_l^{mouse\ isolé}$$

Nous observons ainsi non plus la position du mot, mais la **direction dans laquelle chaque contexte le déplace**. La norme du delta indique l'ampleur du déplacement ; le cosinus compare leurs directions.

In [ ]:
# Trajectoire de référence : le token `mouse` sans contexte lexical.
encoded, _, _, indices = prepare("mouse", "mouse")
with torch.inference_mode():
    inputs = {name: value.to(device) for name, value in encoded.items()}
    output = model(**inputs, output_hidden_states=True, return_dict=True)
    isolated_mouse = torch.stack([
        state[0, indices, :].mean(dim=0).cpu()
        for state in output.hidden_states
    ])

deltas = {context: trajectory - isolated_mouse
          for context, trajectory in mouse_trajectories.items()}
delta_vectors = np.vstack([deltas[c].numpy() for c in contexts])
delta_pca = PCA(n_components=2).fit(delta_vectors)
projected_origin = delta_pca.transform(np.zeros((1, delta_vectors.shape[1])))[0]
delta_xy = {c: delta_pca.transform(deltas[c].numpy()) - projected_origin
            for c in contexts}
norms = {c: torch.linalg.vector_norm(deltas[c], dim=1).numpy() for c in contexts}
separation = torch.linalg.vector_norm(deltas[contexts[0]] - deltas[contexts[1]], dim=1).numpy()
similarity = cosine_similarity(deltas[contexts[0]], deltas[contexts[1]], dim=1).numpy()
angles = np.degrees(np.arccos(np.clip(similarity, -1, 1)))

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "xy"}, {"type": "xy", "secondary_y": True}]],
    column_widths=[0.56, 0.44], horizontal_spacing=0.10,
    subplot_titles=("Deux flèches depuis le même point",
                    "Angle, longueurs et séparation"),
)

def arrow_trace(context, layer):
    x, y = delta_xy[context][layer]
    return go.Scatter(
        x=[0, x], y=[0, y], mode="lines+markers", name=context,
        line=dict(color=colors[context], width=5),
        marker=dict(color=colors[context], size=[6, 12]),
        customdata=[[labels[layer], norms[context][layer]]] * 2,
        hovertemplate=("<b>%{customdata[0]}</b><br>" + context +
                       "<br>Longueur=%{customdata[1]:.2f}<extra></extra>"),
    )

def connector_trace(layer):
    a, b = (delta_xy[c][layer] for c in contexts)
    return go.Scatter(
        x=[a[0], b[0]], y=[a[1], b[1]], mode="lines",
        line=dict(color="#64748b", width=2, dash="dot"),
        name="Séparation", showlegend=False, hoverinfo="skip",
    )

for context in contexts:
    fig.add_trace(arrow_trace(context, 0), row=1, col=1)
fig.add_trace(connector_trace(0), row=1, col=1)

fig.add_trace(go.Scatter(
    x=layers, y=angles, mode="lines+markers", name="Angle",
    line=dict(color="#7c3aed", width=3),
    hovertemplate="Couche %{x}<br>Angle=%{y:.1f}°<extra></extra>",
), row=1, col=2, secondary_y=False)
for context in contexts:
    fig.add_trace(go.Scatter(
        x=layers, y=norms[context], mode="lines",
        name=f"Longueur — {context}", line=dict(color=colors[context], width=2),
        hovertemplate="Couche %{x}<br>Longueur=%{y:.2f}<extra></extra>",
    ), row=1, col=2, secondary_y=True)
fig.add_trace(go.Scatter(
    x=layers, y=separation, mode="lines", name="Distance entre les pointes",
    line=dict(color="#475569", width=2, dash="dot"),
    hovertemplate="Couche %{x}<br>Distance=%{y:.2f}<extra></extra>",
), row=1, col=2, secondary_y=True)

fig.frames = [go.Frame(
    name=str(layer),
    data=[arrow_trace(contexts[0], layer),
          arrow_trace(contexts[1], layer),
          connector_trace(layer)],
    traces=[0, 1, 2],
) for layer in layers]
fig.update_layout(sliders=[dict(
    active=0, currentvalue={"prefix": "Couche observée : "},
    steps=[dict(label=str(layer), method="animate",
                args=[[str(layer)], {"mode": "immediate",
                                     "frame": {"duration": 0, "redraw": True},
                                     "transition": {"duration": 0}}])
           for layer in layers],
)])

limit = 1.15 * max(abs(np.vstack(list(delta_xy.values()))).max(), 1e-6)
variance = 100 * delta_pca.explained_variance_ratio_
fig.update_xaxes(title_text=f"PC1 ({variance[0]:.1f} %)",
                 range=[-limit, limit], zeroline=True, row=1, col=1)
fig.update_yaxes(title_text=f"PC2 ({variance[1]:.1f} %)",
                 range=[-limit, limit], scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="État / couche", dtick=1, row=1, col=2)
fig.update_yaxes(title_text="Angle (degrés)", range=[0, 180],
                 row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="Longueur / distance",
                 row=1, col=2, secondary_y=True)
fig.update_layout(
    template="plotly_white", height=650, width=1200,
    title="Comment chaque contexte déplace-t-il `mouse` ?",
    legend=dict(orientation="h", y=1.08, x=0),
)
fig.show()

## Expérience 4 — Une sonde sémantique contextualisée pour `mouse`

Comme pour `avocat`, nous construisons deux prototypes à partir de phrases contenant toutes le même token `mouse` :

- 🐭 plusieurs emplois désignant l'animal ;
- 🖱️ plusieurs emplois désignant le périphérique informatique.

Le score affiché est **affinité animale − affinité informatique** : positif vers l'animal, négatif vers l'informatique. Il s'agit d'une sonde descriptive, pas d'une preuve que cet axe correspond exactement à un concept interne du modèle.

In [ ]:
mouse_prototypes = {
    "🐭 Animal": [
        "A small mouse ran through the house.",
        "The cat chased the mouse.",
        "The mouse hid under the table.",
    ],
    "🖱️ Informatique": [
        "She moved the pointer with the mouse.",
        "This wireless mouse connects to the computer.",
        "He used the mouse to select the file.",
    ],
}

def mouse_contextual_trajectory(sentence):
    encoded, _, _, indices = prepare(sentence, "mouse")
    with torch.inference_mode():
        inputs = {name: value.to(device) for name, value in encoded.items()}
        output = model(**inputs, output_hidden_states=True, return_dict=True)
    return torch.stack([
        state[0, indices, :].mean(dim=0).cpu()
        for state in output.hidden_states
    ])

mouse_prototype_states = {
    family: torch.stack([mouse_contextual_trajectory(sentence)
                         for sentence in examples]).mean(dim=0)
    for family, examples in mouse_prototypes.items()
}
mouse_scores = {
    context: {
        family: cosine_similarity(trajectory, prototype, dim=1).numpy()
        for family, prototype in mouse_prototype_states.items()
    }
    for context, trajectory in mouse_trajectories.items()
}

animal, computer = mouse_prototypes
mouse_preferences = {context: scores[animal] - scores[computer]
                     for context, scores in mouse_scores.items()}
fig = make_subplots(rows=1, cols=2, subplot_titles=list(mouse_sentences.values()))
for col, context in enumerate(contexts, start=1):
    hover = [[labels[i], mouse_scores[context][animal][i],
              mouse_scores[context][computer][i]] for i in layers]
    fig.add_trace(go.Scatter(
        x=layers, y=mouse_preferences[context], mode="lines+markers",
        name=context, showlegend=False, line=dict(color=colors[context], width=3),
        marker=dict(size=8), customdata=hover,
        hovertemplate=("<b>%{customdata[0]}</b><br>" +
                       "Animal=%{customdata[1]:.4f}<br>" +
                       "Informatique=%{customdata[2]:.4f}<br>" +
                       "Préférence=%{y:+.4f}<extra></extra>"),
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=[layers.min(), layers.max()], y=[0, 0], mode="lines",
        line=dict(color="#94a3b8", dash="dot"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=col)

fig.update_xaxes(title_text="État / couche", dtick=1)
fig.update_yaxes(title_text="Préférence : animal − informatique", col=1)
fig.update_yaxes(matches="y", col=2)
fig.update_layout(
    template="plotly_white", height=520, width=1200, hovermode="x unified",
    title="À quel prototype contextualisé ressemble chaque `mouse` ?",
)
fig.show()

## Expérience 5 — La même sonde pour `avocat`

Cette fois, les ancres ne sont plus des mots isolés : ce sont des phrases contenant toutes le même token `avocat`. Nous construisons ainsi deux prototypes contextualisés :

- ⚖️ plusieurs emplois juridiques d'`avocat` ;
- 🥑 plusieurs emplois alimentaires d'`avocat`.

Le score affiché est **affinité juridique − affinité alimentaire**. Une valeur positive favorise le prototype juridique ; une valeur négative favorise le prototype alimentaire.

In [ ]:
avocat_prototypes = {
    "⚖️ Juridique": [
        "L'avocat défend son client.",
        "Cet avocat prépare le procès.",
        "L'avocat conseille la défense.",
    ],
    "🥑 Alimentaire": [
        "Cet avocat est bien mûr.",
        "Je prépare un guacamole avec l'avocat.",
        "Elle coupe l'avocat pour le déjeuner.",
    ],
}

def contextual_trajectory(sentence, word="avocat"):
    encoded, _, _, indices = prepare(sentence, word)
    with torch.inference_mode():
        inputs = {name: value.to(device) for name, value in encoded.items()}
        output = model(**inputs, output_hidden_states=True, return_dict=True)
    return torch.stack([
        state[0, indices, :].mean(dim=0).cpu()
        for state in output.hidden_states
    ])

prototype_states = {
    family: torch.stack([contextual_trajectory(sentence) for sentence in examples]).mean(dim=0)
    for family, examples in avocat_prototypes.items()
}
avocat_scores = {
    context: {
        family: cosine_similarity(trajectory, prototype, dim=1).numpy()
        for family, prototype in prototype_states.items()
    }
    for context, trajectory in trajectories.items()
}

avocat_contexts = list(trajectories)
avocat_layers = np.arange(len(trajectories[avocat_contexts[0]]))
avocat_labels = np.array(["Représentation initiale"] +
                         [f"Couche {i}" for i in avocat_layers[1:]])
legal, food = avocat_prototypes
preferences = {context: scores[legal] - scores[food]
               for context, scores in avocat_scores.items()}
context_colors = {"⚖️ Justice": "#2563eb", "🥑 Cuisine": "#16a34a"}

fig = make_subplots(rows=1, cols=2, subplot_titles=list(sentences.values()))
for col, context in enumerate(avocat_contexts, start=1):
    hover = [[avocat_labels[i], avocat_scores[context][legal][i],
              avocat_scores[context][food][i]] for i in avocat_layers]
    fig.add_trace(go.Scatter(
        x=avocat_layers, y=preferences[context], mode="lines+markers",
        name=context, showlegend=False, line=dict(color=context_colors[context], width=3),
        marker=dict(size=8), customdata=hover,
        hovertemplate=("<b>%{customdata[0]}</b><br>" +
                       "Juridique=%{customdata[1]:.4f}<br>" +
                       "Alimentaire=%{customdata[2]:.4f}<br>" +
                       "Préférence=%{y:+.4f}<extra></extra>"),
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=[avocat_layers.min(), avocat_layers.max()], y=[0, 0], mode="lines",
        line=dict(color="#94a3b8", dash="dot"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=col)

fig.update_xaxes(title_text="État / couche", dtick=1)
fig.update_yaxes(title_text="Préférence : juridique − alimentaire", col=1)
fig.update_yaxes(matches="y", col=2)
fig.update_layout(
    template="plotly_white", height=520, width=1200, hovermode="x unified",
    title="À quel prototype contextualisé ressemble chaque `avocat` ?",
)
fig.show()

## Expérience 6 — Ce que chaque couche change

Les courbes précédentes donnent une préférence cumulée. Nous calculons maintenant la contribution apparente de chaque couche :

$$\Delta P_l = P_l - P_{l-1}$$

Une barre positive ou négative indique dans quelle direction notre sonde s'est déplacée pendant cette couche. Elle mesure une **variation détectée par la sonde**, pas une décision consciente ni l'action isolée d'un mécanisme unique.

In [ ]:
contribution_panels = [
    (mouse_sentences[contexts[0]], mouse_preferences[contexts[0]], 1, 1,
     "#2563eb", "#f97316", "+ animal / − informatique"),
    (mouse_sentences[contexts[1]], mouse_preferences[contexts[1]], 1, 2,
     "#2563eb", "#f97316", "+ animal / − informatique"),
    (sentences[avocat_contexts[0]], preferences[avocat_contexts[0]], 2, 1,
     "#2563eb", "#16a34a", "+ juridique / − alimentaire"),
    (sentences[avocat_contexts[1]], preferences[avocat_contexts[1]], 2, 2,
     "#2563eb", "#16a34a", "+ juridique / − alimentaire"),
]

fig = make_subplots(
    rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.10,
    subplot_titles=[panel[0] for panel in contribution_panels],
)
for title, preference, row, col, positive_color, negative_color, direction in contribution_panels:
    changes = np.diff(preference)
    layer_numbers = np.arange(1, len(preference))
    bar_colors = np.where(changes >= 0, positive_color, negative_color)
    hover = [[preference[layer], direction] for layer in layer_numbers]
    fig.add_trace(go.Bar(
        x=layer_numbers, y=changes, marker_color=bar_colors, showlegend=False,
        customdata=hover,
        hovertemplate=("<b>Couche %{x}</b><br>Contribution=%{y:+.4f}<br>" +
                       "Préférence cumulée=%{customdata[0]:+.4f}<br>" +
                       "%{customdata[1]}<extra></extra>"),
    ), row=row, col=col)

fig.update_xaxes(title_text="Couche", dtick=1)
fig.update_yaxes(title_text="Variation de préférence", zeroline=True,
                 zerolinewidth=2, zerolinecolor="#475569")
fig.update_layout(
    template="plotly_white", height=820, width=1200,
    title="Quelles couches renforcent ou réduisent la préférence sémantique ?",
    bargap=0.18,
)
fig.show()

## Expériences 7 et 8 — Deux nouvelles polysémies françaises

Nous appliquons la même sonde contextualisée à deux nouveaux mots :

- `vol` : déplacement d'un oiseau ↔ délit ;
- `voile` : voile d'un bateau ↔ tissu porté par une personne.

Pour chaque sens, le prototype est la moyenne de trois représentations du **même token** dans trois phrases d'ancrage. Le score positif correspond au premier sens annoncé dans le titre, le score négatif au second.

In [ ]:
def contextual_probe(word, test_sentences, prototypes, positive_label, negative_label, colors):
    # Un prototype par sens, construit au même niveau dans les 12 couches.
    prototype_states = {
        sense: torch.stack([contextual_trajectory(sentence, word)
                            for sentence in examples]).mean(dim=0)
        for sense, examples in prototypes.items()
    }
    test_states = {context: contextual_trajectory(sentence, word)
                   for context, sentence in test_sentences.items()}
    scores = {
        context: {sense: cosine_similarity(states, prototype, dim=1).numpy()
                  for sense, prototype in prototype_states.items()}
        for context, states in test_states.items()
    }
    preference = {context: values[positive_label] - values[negative_label]
                  for context, values in scores.items()}

    local_layers = np.arange(len(next(iter(test_states.values()))))
    local_labels = np.array(["Représentation initiale"] +
                            [f"Couche {i}" for i in local_layers[1:]])
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(test_sentences.values()))
    for col, context in enumerate(test_sentences, start=1):
        hover = [[local_labels[i], scores[context][positive_label][i],
                  scores[context][negative_label][i]] for i in local_layers]
        fig.add_trace(go.Scatter(
            x=local_layers, y=preference[context], mode="lines+markers",
            name=context, showlegend=False, line=dict(color=colors[context], width=3),
            marker=dict(size=8), customdata=hover,
            hovertemplate=("<b>%{customdata[0]}</b><br>" +
                           positive_label + "=%{customdata[1]:.4f}<br>" +
                           negative_label + "=%{customdata[2]:.4f}<br>" +
                           "Préférence=%{y:+.4f}<extra></extra>"),
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=[local_layers.min(), local_layers.max()], y=[0, 0], mode="lines",
            line=dict(color="#94a3b8", dash="dot"),
            showlegend=False, hoverinfo="skip",
        ), row=1, col=col)

    fig.update_xaxes(title_text="État / couche", dtick=1)
    fig.update_yaxes(title_text=f"Préférence : {positive_label} − {negative_label}", col=1)
    fig.update_yaxes(matches="y", col=2)
    fig.update_layout(
        template="plotly_white", height=520, width=1200, hovermode="x unified",
        title=f"À quel prototype contextualisé ressemble chaque `{word}` ?",
    )
    return fig


### Expérience 7 — `vol` : voler dans les airs ou commettre un délit

Prototype **aérien** : « Le vol de l'aigle était majestueux », « Le jeune oiseau commence son vol », « Nous observons le vol des hirondelles ».

Prototype **délit** : « La police enquête sur le vol », « Ce vol a eu lieu pendant la nuit », « Le suspect reconnaît le vol ».

In [ ]:
vol_tests = {
    "🕊️ Aérien": "L'oiseau prend son vol.",
    "🚔 Délit": "L'auteur du vol a été arrêté.",
}
vol_prototypes = {
    "Aérien": [
        "Le vol de l'aigle était majestueux.",
        "Le jeune oiseau commence son vol.",
        "Nous observons le vol des hirondelles.",
    ],
    "Délit": [
        "La police enquête sur le vol.",
        "Ce vol a eu lieu pendant la nuit.",
        "Le suspect reconnaît le vol.",
    ],
}
contextual_probe(
    "vol", vol_tests, vol_prototypes, "Aérien", "Délit",
    {"🕊️ Aérien": "#0284c7", "🚔 Délit": "#dc2626"},
).show()

### Expérience 8 — `voile` : navigation ou vêtement

Prototype **navigation** : « Le marin hisse la voile », « Le vent pousse la voile du bateau », « La voile se tend au-dessus du pont ».

Prototype **vêtement** : « Elle ajuste son voile avant de sortir », « Son voile couvre ses cheveux », « La mariée porte un long voile blanc ».

In [ ]:
voile_tests = {
    "⛵ Navigation": "La voile était gonflée par le vent.",
    "🧕 Vêtement": "La femme porte un voile noir.",
}
voile_prototypes = {
    "Navigation": [
        "Le marin hisse la voile.",
        "Le vent pousse la voile du bateau.",
        "La voile se tend au-dessus du pont.",
    ],
    "Vêtement": [
        "Elle ajuste son voile avant de sortir.",
        "Son voile couvre ses cheveux.",
        "La mariée porte un long voile blanc.",
    ],
}
contextual_probe(
    "voile", voile_tests, voile_prototypes, "Navigation", "Vêtement",
    {"⛵ Navigation": "#0f766e", "🧕 Vêtement": "#7c3aed"},
).show()

## Expériences 9 et 10 — Le contexte rapproche-t-il deux espèces ?

Nous comparons maintenant directement deux tokens, couche par couche, dans quatre conditions contrôlées :

1. les mots isolés ;
2. un contexte neutre partagé ;
3. un contexte partagé mais sémantiquement non pertinent ;
4. un contexte taxonomique explicite.

Le contrôle non pertinent est essentiel : il permet de distinguer un rapprochement sémantique d'un simple effet dû aux mots communs et à la structure parallèle des phrases.

In [ ]:
def pair_context_probe(word_a, word_b, conditions, title):
    similarities = {}
    for condition, (sentence_a, sentence_b) in conditions.items():
        states_a = contextual_trajectory(sentence_a, word_a)
        states_b = contextual_trajectory(sentence_b, word_b)
        similarities[condition] = cosine_similarity(states_a, states_b, dim=1).numpy()

    local_layers = np.arange(len(next(iter(similarities.values()))))
    local_labels = np.array(["Représentation initiale"] +
                            [f"Couche {i}" for i in local_layers[1:]])
    isolated_final = similarities["Mots isolés"][-1]
    summary = pd.DataFrame([
        {"Condition": condition,
         "Similarité initiale": values[0],
         "Similarité finale": values[-1],
         "Évolution interne": values[-1] - values[0],
         "Gain final vs mots isolés": values[-1] - isolated_final}
        for condition, values in similarities.items()
    ])
    display(summary.round(4))

    palette = {
        "Mots isolés": "#64748b",
        "Contexte neutre partagé": "#0ea5e9",
        "Contexte non pertinent partagé": "#f59e0b",
        "Contexte taxonomique explicite": "#7c3aed",
    }
    fig = go.Figure()
    for condition, values in similarities.items():
        fig.add_trace(go.Scatter(
            x=local_layers, y=values, mode="lines+markers", name=condition,
            line=dict(color=palette[condition], width=3), marker=dict(size=7),
            customdata=local_labels,
            hovertemplate=("<b>%{customdata}</b><br>" + condition +
                           "<br>Similarité=%{y:.4f}<extra></extra>"),
        ))
    fig.update_layout(
        template="plotly_white", height=540, width=1100, hovermode="x unified",
        title=title, xaxis_title="État / couche",
        yaxis_title=f"Similarité cosinus {word_a}–{word_b}",
        yaxis_range=[-0.05, 1.05], legend=dict(orientation="h", y=1.12, x=0),
    )
    return fig


### Expérience 9 — `chat` et `lion`

L'hypothèse testée est que le contexte « félin doté de griffes rétractiles » rapproche davantage les deux tokens que des phrases simplement parallèles.

In [ ]:
chat_lion_conditions = {
    "Mots isolés": ("chat", "lion"),
    "Contexte neutre partagé": (
        "J'observe ce chat sur une image.",
        "J'observe ce lion sur une image.",
    ),
    "Contexte non pertinent partagé": (
        "Le mot chat est écrit en noir.",
        "Le mot lion est écrit en noir.",
    ),
    "Contexte taxonomique explicite": (
        "Ce chat est un félin doté de griffes rétractiles.",
        "Ce lion est un félin doté de griffes rétractiles.",
    ),
}
pair_context_probe(
    "chat", "lion", chat_lion_conditions,
    "`chat`–`lion` : le contexte félin augmente-t-il leur similarité ?",
).show()

### Expérience 10 — `chien` et `loup`

Cette paire témoin reprend le même protocole avec le contexte « canidé doté de griffes non rétractiles ».

In [ ]:
chien_loup_conditions = {
    "Mots isolés": ("chien", "loup"),
    "Contexte neutre partagé": (
        "J'observe ce chien sur une image.",
        "J'observe ce loup sur une image.",
    ),
    "Contexte non pertinent partagé": (
        "Le mot chien est écrit en noir.",
        "Le mot loup est écrit en noir.",
    ),
    "Contexte taxonomique explicite": (
        "Ce chien est un canidé doté de griffes non rétractiles.",
        "Ce loup est un canidé doté de griffes non rétractiles.",
    ),
}
pair_context_probe(
    "chien", "loup", chien_loup_conditions,
    "`chien`–`loup` : le contexte canidé augmente-t-il leur similarité ?",
).show()

In [ ]:
# ============================================================
# EXPERIMENT — Semantic cohesion from similarities CSV
# Step 1: reconstruct the similarity matrix
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1 — Load latest similarities CSV
# ------------------------------------------------------------

folder = Path(
    r"C:\Users\Phil\Projects\LLM-Visual-Explorer\notebooks\explorations\super_scenario_3cat_fr_30\MiniLM"
)

CSV_PATH = max(
    folder.glob("similarities*.csv"),
    key=lambda p: p.stat().st_mtime
)

print(f"Using latest CSV: {CSV_PATH.name}")

df = pd.read_csv(CSV_PATH)

print(f"CSV loaded: {CSV_PATH}")
print(f"Rows: {len(df)}")


# ------------------------------------------------------------
# 2 — Extract concepts
# ------------------------------------------------------------

concepts = sorted(
    set(df["Concept_A"]).union(df["Concept_B"])
)

n = len(concepts)
expected_pairs = n * (n - 1) // 2

print(f"\nConcepts ({n}):")
print(", ".join(concepts))

print(f"\nPairs found:    {len(df)}")
print(f"Pairs expected: {expected_pairs}")

if len(df) != expected_pairs:
    print("⚠️ WARNING — The CSV does not contain exactly one row per pair.")
else:
    print("✅ Complete pair set")


# ------------------------------------------------------------
# 3 — Reconstruct similarity matrix
# ------------------------------------------------------------

similarity_matrix = pd.DataFrame(
    np.eye(n),
    index=concepts,
    columns=concepts,
)

for _, row in df.iterrows():
    a = row["Concept_A"]
    b = row["Concept_B"]
    similarity = row["Similarity"]

    similarity_matrix.loc[a, b] = similarity
    similarity_matrix.loc[b, a] = similarity


# ------------------------------------------------------------
# 4 — Compute scenario cohesion
# ------------------------------------------------------------

pair_similarities = df["Similarity"].to_numpy()

cohesion = pair_similarities.mean()
dispersion = pair_similarities.std()

print("\n" + "=" * 60)
print("SEMANTIC STRUCTURE")
print("=" * 60)

print(f"Cohesion:   {cohesion:.3f}  ({cohesion:.1%})")
print(f"Dispersion: {dispersion:.3f}")

print("\nSimilarity matrix:")
display(similarity_matrix.round(3))

In [ ]:
# ============================================================
# EXPERIMENT — Hierarchical semantic clustering + ARI
# Step 2: dendrogram from cosine distances
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 1 — Convert similarities to distances
# ------------------------------------------------------------

distance_matrix = 1 - similarity_matrix


# ------------------------------------------------------------
# 2 — Convert square matrix to condensed form
#     required by scipy linkage()
# ------------------------------------------------------------

condensed_distances = squareform(
    distance_matrix.values,
    checks=True,
)


# ------------------------------------------------------------
# 3 — Hierarchical clustering
# ------------------------------------------------------------

Z = linkage(
    condensed_distances,
    method="average",
)


# ------------------------------------------------------------
# 4 — Dendrogram
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

dendrogram(
    Z,
    labels=concepts,
    leaf_rotation=45,
    leaf_font_size=11,
)

plt.title("Hierarchical semantic clustering")
plt.ylabel("Cosine distance")
plt.xlabel("Concepts")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5 — Human reference categories
# ------------------------------------------------------------

true_category = {
    # Animals
    "Chien": "Animal",
    "Lion": "Animal",
    "Dauphin": "Animal",
    "Aigle": "Animal",
    "Cheval": "Animal",
    "Papillon": "Animal",
    "Requin": "Animal",
    "Loup": "Animal",
    "Hibou": "Animal",
    "Fourmi": "Animal",

    # Professions
    "Médecin": "Métier",
    "Ingénieur": "Métier",
    "Avocat": "Métier",
    "Boulanger": "Métier",
    "Professeur": "Métier",
    "Plombier": "Métier",
    "Chercheur": "Métier",
    "Agriculteur": "Métier",
    "Architecte": "Métier",
    "Infirmier": "Métier",

    # Objects
    "Marteau": "Objet",
    "Chaise": "Objet",
    "Voiture": "Objet",
    "Téléphone": "Objet",
    "Livre": "Objet",
    "Couteau": "Objet",
    "Ordinateur": "Objet",
    "Valise": "Objet",
    "Lampe": "Objet",
    "Bouteille": "Objet",
}


# ------------------------------------------------------------
# 6 — Build human labels in the same order as concepts
# ------------------------------------------------------------

missing = [
    concept
    for concept in concepts
    if concept not in true_category
]

if missing:
    raise ValueError(
        "Missing human categories for: "
        + ", ".join(missing)
    )

y_true = [
    true_category[concept]
    for concept in concepts
]


# ------------------------------------------------------------
# 7 — Cut dendrogram into exactly 3 clusters
# ------------------------------------------------------------

y_pred = fcluster(
    Z,
    t=3,
    criterion="maxclust",
)


# ------------------------------------------------------------
# 8 — Adjusted Rand Index
# ------------------------------------------------------------

ari = adjusted_rand_score(
    y_true,
    y_pred,
)

print("\n" + "=" * 60)
print("CLUSTER AGREEMENT WITH HUMAN CATEGORIES")
print("=" * 60)

print(f"Number of concepts: {len(concepts)}")
print("Human categories:   3")
print("Detected clusters:  3")
print(f"ARI:                {ari:.3f}")

print("\nInterpretation:")
print("  1.000  = perfect agreement")
print("  ~0.000 = agreement close to chance")
print("  <0.000 = agreement worse than chance")


# ------------------------------------------------------------
# 9 — Show cluster composition
# ------------------------------------------------------------

cluster_results = pd.DataFrame({
    "Concept": concepts,
    "Human_category": y_true,
    "Cluster": y_pred,
})

cluster_results = cluster_results.sort_values(
    by=["Cluster", "Human_category", "Concept"]
)

print("\nCluster composition:")
display(cluster_results)

In [ ]:
# ============================================================
# EXPERIMENT — ARI profile
# Compare human categories with dendrogram cuts
# for k = 2 ... 15 clusters
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import fcluster
from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 1 — Human reference labels
#     in the same order as concepts
# ------------------------------------------------------------

y_true = [
    true_category[concept]
    for concept in concepts
]


# ------------------------------------------------------------
# 2 — Compute ARI for different numbers of clusters
# ------------------------------------------------------------

results = []

for k in range(2, 16):

    y_pred = fcluster(
        Z,
        t=k,
        criterion="maxclust",
    )

    ari = adjusted_rand_score(
        y_true,
        y_pred,
    )

    # fcluster may occasionally return fewer clusters
    # than requested, depending on the dendrogram.
    actual_k = len(set(y_pred))

    results.append({
        "Requested_clusters": k,
        "Actual_clusters": actual_k,
        "ARI": ari,
    })


# ------------------------------------------------------------
# 3 — Results table
# ------------------------------------------------------------

ari_results = pd.DataFrame(results)

print("\n" + "=" * 60)
print("ARI PROFILE — HUMAN CATEGORIES vs HIERARCHICAL CLUSTERS")
print("=" * 60)

display(
    ari_results.style.format({
        "ARI": "{:.3f}"
    })
)


# ------------------------------------------------------------
# 4 — Find the best agreement
# ------------------------------------------------------------

best_row = ari_results.loc[
    ari_results["ARI"].idxmax()
]

best_k = int(best_row["Requested_clusters"])
best_actual_k = int(best_row["Actual_clusters"])
best_ari = best_row["ARI"]

print(
    f"\nBest ARI: {best_ari:.3f}"
    f"  — requested k = {best_k}"
    f"  — actual clusters = {best_actual_k}"
)


# ------------------------------------------------------------
# 5 — Plot ARI as a function of k
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    ari_results["Requested_clusters"],
    ari_results["ARI"],
    marker="o",
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1,
)

plt.scatter(
    [best_k],
    [best_ari],
    s=100,
    zorder=5,
)

plt.title("Agreement with human categories")
plt.xlabel("Number of hierarchical clusters (k)")
plt.ylabel("Adjusted Rand Index (ARI)")

plt.xticks(range(2, 16))
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT — Inspect best ARI clustering
# Show composition of k = 4 clusters
# ============================================================

import pandas as pd

from scipy.cluster.hierarchy import fcluster


# ------------------------------------------------------------
# 1 — Cut dendrogram into 4 clusters
# ------------------------------------------------------------

k = 4

cluster_labels = fcluster(
    Z,
    t=k,
    criterion="maxclust",
)


# ------------------------------------------------------------
# 2 — Build detailed results table
# ------------------------------------------------------------

cluster_details = pd.DataFrame({
    "Concept": concepts,
    "Human_category": [
        true_category[concept]
        for concept in concepts
    ],
    "Cluster": cluster_labels,
})

cluster_details = cluster_details.sort_values(
    by=["Cluster", "Human_category", "Concept"]
)

print("\n" + "=" * 60)
print(f"CLUSTER COMPOSITION — k = {k}")
print("=" * 60)

display(cluster_details)


# ------------------------------------------------------------
# 3 — Count human categories inside each cluster
# ------------------------------------------------------------

cluster_summary = (
    cluster_details
    .groupby(["Cluster", "Human_category"])
    .size()
    .unstack(fill_value=0)
)


# Add total size of each cluster
cluster_summary["Total"] = cluster_summary.sum(axis=1)

print("\nComposition by human category:")
display(cluster_summary)


# ------------------------------------------------------------
# 4 — Optional: print concepts cluster by cluster
# ------------------------------------------------------------

for cluster_id in sorted(cluster_details["Cluster"].unique()):

    subset = cluster_details[
        cluster_details["Cluster"] == cluster_id
    ]

    print("\n" + "-" * 60)
    print(f"Cluster {cluster_id} — {len(subset)} concepts")
    print("-" * 60)

    for category in ["Animal", "Métier", "Objet"]:

        names = subset.loc[
            subset["Human_category"] == category,
            "Concept"
        ].tolist()

        if names:
            print(
                f"{category:8s}: "
                + ", ".join(names)
            )

In [ ]:
# ============================================================
# EXPERIMENT — Store ARI profile for current model
# Run this cell after computing Z and loading the current CSV
# ============================================================

import pandas as pd

from scipy.cluster.hierarchy import fcluster
from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 1 — Name of current model
#    Change only this line before each run
# ------------------------------------------------------------

CURRENT_MODEL = "MiniLM"
# Examples:
# "MiniLM"
# "MPNet"
# "BERT"
# "LaBSE"


# ------------------------------------------------------------
# 2 — Human reference labels
# ------------------------------------------------------------

y_true = [
    true_category[concept]
    for concept in concepts
]


# ------------------------------------------------------------
# 3 — Compute ARI profile for k = 2 ... 15
# ------------------------------------------------------------

rows = []

for k in range(2, 16):

    y_pred = fcluster(
        Z,
        t=k,
        criterion="maxclust",
    )

    ari = adjusted_rand_score(
        y_true,
        y_pred,
    )

    actual_k = len(set(y_pred))

    rows.append({
        "Model": CURRENT_MODEL,
        "Requested_clusters": k,
        "Actual_clusters": actual_k,
        "ARI": ari,
    })


current_ari_profile = pd.DataFrame(rows)


# ------------------------------------------------------------
# 4 — Create storage dictionary if needed
# ------------------------------------------------------------

if "ari_profiles" not in globals():
    ari_profiles = {}


# ------------------------------------------------------------
# 5 — Store current model profile
# ------------------------------------------------------------

ari_profiles[CURRENT_MODEL] = current_ari_profile.copy()


# ------------------------------------------------------------
# 6 — Show summary for current model
# ------------------------------------------------------------

best_row = current_ari_profile.loc[
    current_ari_profile["ARI"].idxmax()
]

best_k = int(best_row["Requested_clusters"])
best_ari = best_row["ARI"]

print("\n" + "=" * 60)
print(f"ARI PROFILE STORED — {CURRENT_MODEL}")
print("=" * 60)

print(f"Best ARI: {best_ari:.3f}")
print(f"Best k:   {best_k}")

print("\nProfiles currently stored:")
for model_name in ari_profiles:
    print(f"  - {model_name}")

display(
    current_ari_profile[
        ["Requested_clusters", "Actual_clusters", "ARI"]
    ].style.format({
        "ARI": "{:.3f}"
    })
)

In [ ]:
# ============================================================
# EXPERIMENT — Compare ARI profiles across models
# ============================================================

import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))


# ------------------------------------------------------------
# 1 — Plot every stored model
# ------------------------------------------------------------

for model_name, profile in ari_profiles.items():

    plt.plot(
        profile["Requested_clusters"],
        profile["ARI"],
        marker="o",
        label=model_name,
    )


# ------------------------------------------------------------
# 2 — Reference lines
# ------------------------------------------------------------

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1,
)

plt.axvline(
    x=3,
    linestyle="--",
    linewidth=1,
)


# ------------------------------------------------------------
# 3 — Common scale
# ------------------------------------------------------------

plt.xlim(2, 15)
plt.ylim(-0.05, 0.40)

plt.xticks(range(2, 16))


# ------------------------------------------------------------
# 4 — Labels
# ------------------------------------------------------------

plt.title("Agreement with human categories — model comparison")
plt.xlabel("Number of hierarchical clusters (k)")
plt.ylabel("Adjusted Rand Index (ARI)")

plt.legend(title="Model")
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT — Contextual meaning across layers
# Multilingual ambiguous words: avocat / mouse / banco
# Model: LaBSE
# ============================================================

# ============================================================
# 1 — Load LaBSE for contextual layer analysis
# ============================================================

import torch

from transformers import AutoTokenizer, AutoModel


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_ID = "sentence-transformers/LaBSE"


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")


# ------------------------------------------------------------
# Tokenizer + model
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID
).to(device)

model.eval()


# ------------------------------------------------------------
# Model information
# ------------------------------------------------------------

print(f"Model: {MODEL_ID}")
print(f"Architecture: {model.config.model_type}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Dimensions: {model.config.hidden_size}")

In [ ]:
# ============================================================
# 2 — Tokenization of ambiguous words
# avocat / mouse / banco
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# Generic preparation function
# ------------------------------------------------------------

def prepare(sentence, word):

    encoded = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()

    tokens = tokenizer.convert_ids_to_tokens(
        encoded["input_ids"][0]
    )

    start = sentence.lower().index(word.lower())
    end = start + len(word)

    indices = [
        i
        for i, (a, b) in enumerate(offsets)
        if b > start and a < end
    ]

    if not indices:
        raise ValueError(
            f"Le mot {word!r} n'a pas été retrouvé."
        )

    return encoded, tokens, offsets, indices


# ------------------------------------------------------------
# One simple sentence for each ambiguous word
# ------------------------------------------------------------

tokenization_tests = {
    "🇫🇷 avocat": {
        "word": "avocat",
        "sentence": "L'avocat défend son client.",
    },

    "🇬🇧 mouse": {
        "word": "mouse",
        "sentence": "The mouse ran through the house.",
    },

    "🇪🇸 banco": {
        "word": "banco",
        "sentence": "Fui al banco para sacar dinero.",
    },
}


# ------------------------------------------------------------
# Analyze tokenization
# ------------------------------------------------------------

rows = []

for example, data in tokenization_tests.items():

    word = data["word"]
    sentence = data["sentence"]

    encoded, tokens, offsets, indices = prepare(
        sentence,
        word,
    )

    for position, (token, offset) in enumerate(
        zip(tokens, offsets)
    ):

        rows.append({
            "Exemple": example,
            "Position": position,
            "Token": token,
            "Caractères": str(tuple(offset)),
            "Mot suivi": (
                f"← {word}"
                if position in indices
                else ""
            ),
        })


tokenization_df = pd.DataFrame(rows)

display(tokenization_df)

In [ ]:
# ============================================================
# 3 — Ambiguous-word experiments
# Contexts + semantic prototypes
# ============================================================


experiments = {

    # --------------------------------------------------------
    # 🇫🇷 AVOCAT
    # --------------------------------------------------------

    "avocat": {

        "word": "avocat",

        "contexts": {
            "⚖️ Justice": "L'avocat plaide devant le tribunal.",
            "🥑 Cuisine": "Elle ajoute de l'avocat dans la salade.",
        },

        "prototypes": {

            "⚖️ Juridique": [
                "L'avocat défend son client.",
                "Cet avocat prépare le procès.",
                "L'avocat conseille la défense.",
            ],

            "🥑 Alimentaire": [
                "Cet avocat est bien mûr.",
                "Je prépare un guacamole avec l'avocat.",
                "Elle coupe l'avocat pour le déjeuner.",
            ],
        },

        "preference_label":
            "Préférence : juridique − alimentaire",
    },


    # --------------------------------------------------------
    # 🇬🇧 MOUSE
    # --------------------------------------------------------

    "mouse": {

        "word": "mouse",

        "contexts": {
            "🐭 Animal": "The mouse ran across the floor.",
            "🖱️ Computer": "She clicked the icon with the mouse.",
        },

        "prototypes": {

            "🐭 Animal": [
                "A small mouse ran through the house.",
                "The cat chased the mouse.",
                "The mouse hid under the table.",
            ],

            "🖱️ Computer": [
                "She moved the pointer with the mouse.",
                "This wireless mouse connects to the computer.",
                "He used the mouse to select the file.",
            ],
        },

        "preference_label":
            "Preference: animal − computer",
    },


    # --------------------------------------------------------
    # 🇪🇸 BANCO
    # --------------------------------------------------------

    "banco": {

        "word": "banco",

        "contexts": {
            "🏦 Finanzas": "Fui al banco para sacar dinero.",
            "🪑 Parque": "Me senté en un banco del parque.",
        },

        "prototypes": {

            "🏦 Financiero": [
                "El banco presta dinero a sus clientes.",
                "Deposité mis ahorros en el banco.",
                "El banco aprobó el préstamo.",
            ],

            "🪑 Asiento": [
                "Me senté en un banco del parque.",
                "El banco de madera está junto al árbol.",
                "Nos sentamos juntos en un banco.",
            ],
        },

        "preference_label":
            "Preferencia: financiero − asiento",
    },
}


# ------------------------------------------------------------
# Quick check
# ------------------------------------------------------------

for name, experiment in experiments.items():

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Contexts:")
    for label, sentence in experiment["contexts"].items():
        print(f"  {label}: {sentence}")

    print("\nPrototypes:")
    for family, examples in experiment["prototypes"].items():
        print(f"  {family}: {len(examples)} examples")

In [ ]:
# ============================================================
# 4 — Generic contextual trajectories across layers
# ============================================================

import numpy as np
import torch

from torch.nn.functional import cosine_similarity


# ------------------------------------------------------------
# 1 — Generic trajectory for one word in one sentence
# ------------------------------------------------------------

def contextual_trajectory(sentence, word):

    encoded, _, _, indices = prepare(
        sentence,
        word,
    )

    with torch.inference_mode():

        inputs = {
            name: value.to(device)
            for name, value in encoded.items()
        }

        output = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    # One vector per state:
    # initial representation + one vector per Transformer layer
    trajectory = torch.stack([
        state[0, indices, :].mean(dim=0).cpu()
        for state in output.hidden_states
    ])

    return trajectory


# ------------------------------------------------------------
# 2 — Compute trajectories, prototypes and preferences
#     for every ambiguous word
# ------------------------------------------------------------

results = {}

for experiment_name, experiment in experiments.items():

    word = experiment["word"]
    contexts = experiment["contexts"]
    prototypes = experiment["prototypes"]

    print("\n" + "=" * 60)
    print(f"PROCESSING: {experiment_name.upper()}")
    print("=" * 60)

    # --------------------------------------------------------
    # Context trajectories
    # --------------------------------------------------------

    trajectories = {
        context_name: contextual_trajectory(
            sentence,
            word,
        )
        for context_name, sentence in contexts.items()
    }

    # --------------------------------------------------------
    # Prototype trajectories
    # --------------------------------------------------------

    prototype_states = {
        family: torch.stack([
            contextual_trajectory(sentence, word)
            for sentence in examples
        ]).mean(dim=0)

        for family, examples in prototypes.items()
    }

    # --------------------------------------------------------
    # Similarity with each prototype at each layer
    # --------------------------------------------------------

    scores = {
        context_name: {
            family: cosine_similarity(
                trajectory,
                prototype,
                dim=1,
            ).numpy()

            for family, prototype in prototype_states.items()
        }

        for context_name, trajectory in trajectories.items()
    }

    # --------------------------------------------------------
    # Semantic preference
    # first prototype minus second prototype
    # --------------------------------------------------------

    prototype_names = list(prototypes.keys())

    positive_family = prototype_names[0]
    negative_family = prototype_names[1]

    preferences = {
        context_name:
            context_scores[positive_family]
            - context_scores[negative_family]

        for context_name, context_scores in scores.items()
    }

    # --------------------------------------------------------
    # Store everything
    # --------------------------------------------------------

    results[experiment_name] = {
        "trajectories": trajectories,
        "prototype_states": prototype_states,
        "scores": scores,
        "preferences": preferences,
        "positive_family": positive_family,
        "negative_family": negative_family,
    }

    print(
        f"States observed: "
        f"{len(next(iter(trajectories.values())))}"
    )

    print(
        f"Preference: "
        f"{positive_family} − {negative_family}"
    )

In [ ]:
# ============================================================
# 5 — Plot contextual preference trajectories
# avocat / mouse / banco
# ============================================================

import numpy as np
import plotly.graph_objects as go

from plotly.subplots import make_subplots


# ------------------------------------------------------------
# Shared layer labels
# ------------------------------------------------------------

layers = np.arange(13)

layer_labels = np.array(
    ["Représentation initiale"] +
    [f"Couche {i}" for i in layers[1:]]
)


# ------------------------------------------------------------
# Plot one experiment at a time
# ------------------------------------------------------------

for experiment_name, experiment in experiments.items():

    experiment_results = results[experiment_name]

    contexts = list(experiment["contexts"].keys())

    positive_family = experiment_results["positive_family"]
    negative_family = experiment_results["negative_family"]

    scores = experiment_results["scores"]
    preferences = experiment_results["preferences"]

    # --------------------------------------------------------
    # Create side-by-side plots
    # --------------------------------------------------------

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            experiment["contexts"][context]
            for context in contexts
        ],
    )

    # --------------------------------------------------------
    # Add one context per subplot
    # --------------------------------------------------------

    for col, context in enumerate(contexts, start=1):

        hover = [
            [
                layer_labels[i],
                scores[context][positive_family][i],
                scores[context][negative_family][i],
            ]
            for i in layers
        ]

        fig.add_trace(
            go.Scatter(
                x=layers,
                y=preferences[context],
                mode="lines+markers",
                name=context,
                showlegend=False,
                marker=dict(size=8),
                line=dict(width=3),
                customdata=hover,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    + f"{positive_family}=%{{customdata[1]:.4f}}<br>"
                    + f"{negative_family}=%{{customdata[2]:.4f}}<br>"
                    + "Préférence=%{y:+.4f}"
                    + "<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )

        # Zero preference reference line
        fig.add_trace(
            go.Scatter(
                x=[layers.min(), layers.max()],
                y=[0, 0],
                mode="lines",
                line=dict(
                    dash="dot",
                    width=1,
                ),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1,
            col=col,
        )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    fig.update_xaxes(
        title_text="État / couche",
        dtick=1,
    )

    fig.update_yaxes(
        title_text=experiment["preference_label"],
        col=1,
    )

    fig.update_yaxes(
        matches="y",
        col=2,
    )

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    fig.update_layout(
        template="plotly_white",
        height=520,
        width=1200,
        hovermode="x unified",
        title=(
            f"À quel prototype contextualisé ressemble "
            f"`{experiment['word']}` ?"
        ),
    )

    fig.show()

In [ ]:
###########################################################################################
# Test de tokenization de avocat
###########################################################################################

from transformers import AutoTokenizer

XLMR_ID = "xlm-roberta-base"

xlmr_tokenizer = AutoTokenizer.from_pretrained(
    XLMR_ID,
    use_fast=True,
)

sentence = "Philippe et Nicole ont une visio avec Alhéli et Adrien et parlent d'Aldo, D'Adanari et Lilith"

encoded = xlmr_tokenizer(
    sentence,
    return_tensors="pt",
    return_offsets_mapping=True,
)

offsets = encoded["offset_mapping"][0].tolist()

tokens = xlmr_tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

for i, (token, offset) in enumerate(zip(tokens, offsets)):
    print(f"{i:2d}  {token:15s}  {tuple(offset)}")





In [ ]:
# 2° test tokeization avocat


word = "avocat"

start = sentence.lower().index(word.lower())
end = start + len(word)

indices = [
    i
    for i, (a, b) in enumerate(offsets)
    if b > start and a < end
]

print("\nTokens correspondant à 'avocat' :")

for i in indices:
    print(i, tokens[i], offsets[i])

In [ ]:
# ============================================================
# DENDROGRAM 0A
# EXPERIMENT — Grim semantic map
# Step 1: reconstruct similarity matrix from CSV
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1 — Select experiment
# ------------------------------------------------------------

SCENARIO = "grim_3x6_en"
MODEL_ALIAS = "MiniLM"   # MiniLM / MPNet / LaBSE / BERT

# ------------------------------------------------------------
# Locate experiment folder
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

folder = (
    PROJECT_ROOT
    / "notebooks"
    / "explorations"
    / SCENARIO
    / MODEL_ALIAS
)

csv_files = list(folder.glob("similarities*.csv"))

print(f"Experiment folder: {folder}")

if not csv_files:
    raise FileNotFoundError(
        f"No similarity CSV found in:\n{folder}"
    )

CSV_PATH = max(
    csv_files,
    key=lambda p: p.stat().st_mtime
)
print(f"Using latest CSV: {CSV_PATH.name}")

df = pd.read_csv(CSV_PATH)

print(f"CSV loaded: {CSV_PATH}")
print(f"Rows: {len(df)}")


# ------------------------------------------------------------
# 2 — Read experiment metadata
# ------------------------------------------------------------

scenario_name = df["Scenario"].iloc[0]
model_alias = df["Model_Alias"].iloc[0]
model_name = df["Model_Name"].iloc[0]
run_date = df["Date"].iloc[0]
run_time = df["Time"].iloc[0]

print()
print(f"Scenario: {scenario_name}")
print(f"Model:    {model_alias}")
print(f"          {model_name}")
print(f"Run:      {run_date} {run_time}")


# ------------------------------------------------------------
# 3 — Extract concepts
# ------------------------------------------------------------

concepts = sorted(
    set(df["Concept_A"]).union(df["Concept_B"])
)

n = len(concepts)
expected_pairs = n * (n - 1) // 2

print(f"\nConcepts ({n}):")
print(", ".join(concepts))

print(f"\nPairs found:    {len(df)}")
print(f"Pairs expected: {expected_pairs}")

if len(df) != expected_pairs:
    print("⚠️ WARNING — The CSV does not contain exactly one row per pair.")
else:
    print("✅ Complete pair set")


# ------------------------------------------------------------
# 4 — Reconstruct similarity matrix
# ------------------------------------------------------------

similarity_matrix = pd.DataFrame(
    np.eye(n),
    index=concepts,
    columns=concepts,
)

for _, row in df.iterrows():
    a = row["Concept_A"]
    b = row["Concept_B"]
    similarity = row["Similarity"]

    similarity_matrix.loc[a, b] = similarity
    similarity_matrix.loc[b, a] = similarity


# ------------------------------------------------------------
# 5 — Compute scenario structure
# ------------------------------------------------------------

pair_similarities = df["Similarity"].to_numpy()

cohesion = pair_similarities.mean()
dispersion = pair_similarities.std()

strongest = df.loc[df["Similarity"].idxmax()]
weakest = df.loc[df["Similarity"].idxmin()]

print("\n" + "=" * 60)
print("SEMANTIC STRUCTURE")
print("=" * 60)

print(f"Concepts:        {n}")
print(f"Mean cohesion:   {cohesion:.3f}  ({cohesion:.1%})")
print(f"Dispersion:      {dispersion:.3f}")

print(
    f"\nStrongest pair:  "
    f"{strongest['Concept_A']} ↔ {strongest['Concept_B']} "
    f"({strongest['Similarity']:.1%})"
)

print(
    f"Weakest pair:    "
    f"{weakest['Concept_A']} ↔ {weakest['Concept_B']} "
    f"({weakest['Similarity']:.1%})"
)

print("\nSimilarity matrix:")
display(similarity_matrix.round(3))

In [ ]:
# ============================================================
# DENDROGRAM 0B
# EXPERIMENT — Grim semantic map
# Step 2: hierarchical clustering
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score


# ------------------------------------------------------------
# 1 — Convert similarities to cosine distances
# ------------------------------------------------------------

distance_matrix = 1 - similarity_matrix


# ------------------------------------------------------------
# 2 — Convert to condensed form
# ------------------------------------------------------------

condensed_distances = squareform(
    distance_matrix.values,
    checks=True,
)


# ------------------------------------------------------------
# 3 — Hierarchical clustering
# ------------------------------------------------------------

Z = linkage(
    condensed_distances,
    method="average",
)


# ------------------------------------------------------------
# 4 — Dendrogram
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

dendrogram(
    Z,
    labels=concepts,
    leaf_rotation=45,
    leaf_font_size=11,
)

plt.title(
    f"Dendrograma semántico\n"
    f"{model_alias} · {scenario_name} · {len(concepts)} conceptos · {run_date}"
)

plt.ylabel("Distancia coseno")
plt.xlabel("Conceptos")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

# =================================================
# Category cohesion / dispersion by language
# =================================================

EXPLORATIONS = Path(
    r"C:\Users\Phil\Projects\LLM-Visual-Explorer\notebooks\explorations"
)

SCENARIOS = {
    "ES": "motivation_conf_es",
    "FR": "motivational_conf_fr",
    "EN": "motivational_conf_en",
}

MODELS = ["MiniLM", "MPNet", "LaBSE", "BERT"]

FRUITS = {
    "ES": {"Mango", "Albaricoque", "Arándano", "Uva"},
    "FR": {"Mangue", "Abricot", "Myrtille", "Raisin"},
    "EN": {"Mango", "Apricot", "Blueberry", "Grape"},
}


def analyse_csv(csv_path, language):
    df = pd.read_csv(csv_path)

    fruits = FRUITS[language]

    def pair_category(row):
        a_is_fruit = row["Concept_A"] in fruits
        b_is_fruit = row["Concept_B"] in fruits

        if a_is_fruit and b_is_fruit:
            return "Fruit ↔ Fruit"

        elif not a_is_fruit and not b_is_fruit:
            return "Animal ↔ Animal"

        else:
            return "Fruit ↔ Animal"

    df["Category"] = df.apply(pair_category, axis=1)

    stats = (
        df.groupby("Category")["Similarity"]
        .agg(["mean", "std", "count"])
    )

    fruit_mean = stats.loc["Fruit ↔ Fruit", "mean"]
    animal_mean = stats.loc["Animal ↔ Animal", "mean"]
    cross_mean = stats.loc["Fruit ↔ Animal", "mean"]

    separation = (
        (fruit_mean + animal_mean) / 2
        - cross_mean
    )

    return {
        "Fruit cohesion": fruit_mean,
        "Fruit dispersion": stats.loc["Fruit ↔ Fruit", "std"],
        "Animal cohesion": animal_mean,
        "Animal dispersion": stats.loc["Animal ↔ Animal", "std"],
        "Cross-category": cross_mean,
        "Cross dispersion": stats.loc["Fruit ↔ Animal", "std"],
        "Separation margin": separation,
    }


# =================================================
# Read latest CSV for each language/model
# =================================================

results = []

for language, scenario in SCENARIOS.items():

    for model in MODELS:

        folder = EXPLORATIONS / scenario / model

        csv_files = list(folder.glob("similarities*.csv"))

        if not csv_files:
            print(f"⚠ No CSV: {language} / {model}")
            continue

        # If several CSVs exist, use the most recent one
        csv_path = max(
            csv_files,
            key=lambda p: p.stat().st_mtime
        )

        stats = analyse_csv(
            csv_path,
            language
        )

        results.append(
            {
                "Language": language,
                "Model": model,
                **stats,
            }
        )


# =================================================
# Final table
# =================================================

results_df = pd.DataFrame(results)

numeric_columns = [
    "Fruit cohesion",
    "Fruit dispersion",
    "Animal cohesion",
    "Animal dispersion",
    "Cross-category",
    "Cross dispersion",
    "Separation margin",
]

# Convert to %
results_df[numeric_columns] *= 100

results_df = results_df.round(1)

display(results_df)

In [ ]:
MODEL_PROFILES = {
    "MiniLM": {
        "alias": "MiniLM",
        "name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "type": "embedding",
        "representation_mode": "sentence_embedding",
        "predictive_state_suffix": "",
    },

    "MPNet": {
        "alias": "MPNet",
        "name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "type": "embedding",
        "representation_mode": "sentence_embedding",
        "predictive_state_suffix": "",
    },

    "BERT": {
        "alias": "BERT",
        "name": "sentence-transformers/bert-base-nli-mean-tokens",
        "type": "embedding",
        "representation_mode": "sentence_embedding",
        "predictive_state_suffix": "",
    },

    "LaBSE": {
        "alias": "LaBSE",
        "name": "sentence-transformers/LaBSE",
        "type": "embedding",
        "representation_mode": "sentence_embedding",
        "predictive_state_suffix": "",
    },
}

In [ ]:
# =================================================
# Model selection
# =================================================

import ipywidgets as widgets
from IPython.display import display

model_selector = widgets.Dropdown(
    options=[
        ("MiniLM — fast & compact", "MiniLM"),
        ("MPNet — semantic representation", "MPNet"),
        ("BERT — historical reference", "BERT"),
        ("LaBSE — multilingual", "LaBSE"),
    ],
    value="MiniLM",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="430px"),
)

display(model_selector)


In [ ]:
from explorer.embeddings import compute_embeddings

embeddings = compute_embeddings(["chien", "chat"])

print("Active model :", config.ACTIVE_MODEL)
print("Shape :", embeddings.shape)

In [ ]:
# ------------------------------------------------------------
# 4 — Simplified pedagogical dendrogram
# ------------------------------------------------------------

from matplotlib.lines import Line2D

# Human categories for this Grim 3x6 experiment
categories = {
    # Fruits
    "Mango": "Fruits",
    "Apricot": "Fruits",
    "Blueberry": "Fruits",
    "Grape": "Fruits",
    "Apple": "Fruits",
    "Orange": "Fruits",

    # Animals
    "Bear": "Animals",
    "Cow": "Animals",
    "Dog": "Animals",
    "Wolf": "Animals",
    "Alligator": "Animals",
    "Crocodile": "Animals",

    # Rock artists
    "Beatles": "Rock",
    "Rolling Stones": "Rock",
    "Nirvana": "Rock",
    "AC/DC": "Rock",
    "Elvis": "Rock",
    "Pink Floyd": "Rock",
}
"""
categories = {
    # Fruits
    "Mango": "Fruits",
    "Albaricoque": "Fruits",
    "Arándano": "Fruits",
    "Uva": "Fruits",
    "Manzana": "Fruits",
    "Naranja": "Fruits",

    # Animals
    "Oso": "Animals",
    "Vaca": "Animals",
    "Perro": "Animals",
    "Lobo": "Animals",
    "Aligátor": "Animals",
    "Cocodrilo": "Animals",

    # Rock
    "Beatles": "Rock",
    "Rolling Stones": "Rock",
    "Nirvana": "Rock",
    "AC/DC": "Rock",
    "Elvis": "Rock",
    "Pink Floyd": "Rock",
}
"""
# Temporary colors for the lab prototype
category_colors = {
    "Fruits": "tab:green",
    "Animals": "tab:orange",
    "Rock": "tab:purple",
}


fig, ax = plt.subplots(figsize=(11, 6))

dendrogram(
    Z,
    labels=concepts,
    leaf_rotation=45,
    leaf_font_size=11,

    # Neutral branches:
    color_threshold=0,
    above_threshold_color="0.45",
    link_color_func=lambda k: "0.45",

    ax=ax,
)

# ------------------------------------------------------------
# Color concept labels according to HUMAN categories
# ------------------------------------------------------------

for label in ax.get_xticklabels():

    concept = label.get_text()
    category = categories.get(concept)

    if category:
        label.set_color(category_colors[category])
        label.set_fontweight("bold")


# ------------------------------------------------------------
# Simplified title and axes
# ------------------------------------------------------------

ax.set_title(
    f"Hierarchical view — {scenario_name}\n"
    f"{model_alias} · {len(concepts)} concepts"
)

ax.set_ylabel("Semantic distance")
ax.set_xlabel("")


# ------------------------------------------------------------
# Human-category legend
# ------------------------------------------------------------

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label=category,
        markerfacecolor=color,
        markeredgecolor=color,
        markersize=8,
    )
    for category, color in category_colors.items()
]

ax.legend(
    handles=legend_handles,
    title="Human categories",
    loc="upper right",
    frameon=False,
)


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT — Progressive dendrogram
# V1: Step-by-step construction
# ============================================================

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from scipy.cluster.hierarchy import dendrogram


# ------------------------------------------------------------
# 1 — Get the final dendrogram geometry WITHOUT drawing it
# ------------------------------------------------------------

ddata = dendrogram(
    Z,
    labels=concepts,
    no_plot=True,
)

# Leaf order chosen by the dendrogram
leaf_labels = ddata["ivl"]

# x positions used internally by scipy:
# 5, 15, 25, 35, ...
leaf_x = {
    label: 5 + 10 * i
    for i, label in enumerate(leaf_labels)
}


# ------------------------------------------------------------
# 2 — State
# ------------------------------------------------------------

state = {
    "step": 0
}

total_steps = len(Z)      # 17 merges for 18 concepts

# ------------------------------------------------------------
# Describe what each linkage node contains
# ------------------------------------------------------------

n_concepts = len(concepts)

# Initial nodes 0 ... n-1 are individual concepts
cluster_members = {
    i: [concepts[i]]
    for i in range(n_concepts)
}

# New nodes n, n+1, ... are created by the rows of Z
for i, row in enumerate(Z):

    left_id = int(row[0])
    right_id = int(row[1])

    new_id = n_concepts + i

    cluster_members[new_id] = (
        cluster_members[left_id]
        + cluster_members[right_id]
    )


def node_is_concept(node_id):
    return node_id < n_concepts


def describe_node(node_id):
    """Human-readable description of a concept or cluster."""

    members = cluster_members[node_id]

    if len(members) == 1:
        return members[0]

    return "{" + ", ".join(members) + "}"

# ------------------------------------------------------------
# 3 — Drawing function
# ------------------------------------------------------------

def draw_progressive_dendrogram():

    step = state["step"]

    fig, ax = plt.subplots(figsize=(11, 6))

    # --------------------------------------------------------
    # Draw only the first N dendrogram links
    # --------------------------------------------------------

    for i in range(step):

        icoord = ddata["icoord"][i]
        dcoord = ddata["dcoord"][i]

        ax.plot(
            icoord,
            dcoord,
            color="0.45",
            linewidth=1.6,
        )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_xlim(0, 10 * len(leaf_labels))

    max_distance = max(max(d) for d in ddata["dcoord"])
    ax.set_ylim(0, max_distance * 1.08)

    ax.set_xticks(
        [leaf_x[label] for label in leaf_labels]
    )

    ax.set_xticklabels(
        leaf_labels,
        rotation=45,
        ha="right",
        fontsize=11,
    )

    # --------------------------------------------------------
    # Human-category colors
    # --------------------------------------------------------

    for label in ax.get_xticklabels():

        concept = label.get_text()
        category = categories.get(concept)

        if category:
            label.set_color(category_colors[category])
            label.set_fontweight("bold")

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        f"Building the hierarchy — {scenario_name}\n"
        f"{model_alias} · Step {step}/{total_steps}"
    )

    ax.set_ylabel("Semantic distance")

        # --------------------------------------------------------
    # Narrative explanation of the current merge
    # --------------------------------------------------------

    if step == 0:

        message = (
            "Start with isolated concepts.\n"
            "Press Next to reveal the first merge."
        )

    else:

        # Row corresponding to the merge just revealed
        row = Z[step - 1]

        left_id = int(row[0])
        right_id = int(row[1])
        distance = row[2]

        left_is_concept = node_is_concept(left_id)
        right_is_concept = node_is_concept(right_id)

        left_text = describe_node(left_id)
        right_text = describe_node(right_id)

        # What kind of merge is happening?
        if left_is_concept and right_is_concept:

            merge_type = "Two concepts join"

        elif left_is_concept or right_is_concept:

            merge_type = "A concept joins a group"

        else:

            merge_type = "Two groups join"

        message = (
            f"Step {step}/{total_steps} — {merge_type}\n"
            f"{left_text}  +  {right_text}\n"
            f"Semantic distance: {distance:.3f}"
        )

    ax.text(
        0.01,
        0.98,
        message,
        transform=ax.transAxes,
        va="top",
        fontsize=11,
        linespacing=1.4,
    )


# ------------------------------------------------------------
# 4 — Controls + refresh
# ------------------------------------------------------------

from IPython.display import display, clear_output


previous_button = widgets.Button(
    description="◀ Previous"
)

next_button = widgets.Button(
    description="Next ▶",
    button_style="primary",
)

reset_button = widgets.Button(
    description="↺ Reset"
)


def refresh():

    clear_output(wait=True)

    controls = widgets.HBox([
        previous_button,
        next_button,
        reset_button,
    ])

    display(controls)

    draw_progressive_dendrogram()


def previous_step(_):

    if state["step"] > 0:
        state["step"] -= 1

    refresh()


def next_step(_):

    if state["step"] < total_steps:
        state["step"] += 1

    refresh()


def reset_steps(_):

    state["step"] = 0

    refresh()


previous_button.on_click(previous_step)
next_button.on_click(next_step)
reset_button.on_click(reset_steps)


# ------------------------------------------------------------
# 5 — First display
# ------------------------------------------------------------

refresh()

In [ ]:
# ============================================================
# EXPERIMENT — Progressive dendrogram
# V2: Step-by-step construction, synchronized narration
# ============================================================

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.cluster.hierarchy import dendrogram


# ------------------------------------------------------------
# 1 — Get the FINAL leaf order
#
# We use scipy only to determine the final left-to-right order.
# The progressive tree itself will be reconstructed directly
# from Z so that drawing and narration always describe the
# exact same merge.
# ------------------------------------------------------------

ddata = dendrogram(
    Z,
    labels=concepts,
    no_plot=True,
)

leaf_labels = ddata["ivl"]

# scipy convention:
# leaf x positions are 5, 15, 25, 35, ...
leaf_x = {
    label: 5 + 10 * i
    for i, label in enumerate(leaf_labels)
}


# ------------------------------------------------------------
# 2 — Build the hierarchy directly from Z
# ------------------------------------------------------------

n_concepts = len(concepts)
total_steps = len(Z)

# Members contained in every node.
#
# Initial nodes:
#     0 ... n-1       individual concepts
#
# Linkage nodes:
#     n, n+1, ...     groups created by Z
cluster_members = {
    i: [concepts[i]]
    for i in range(n_concepts)
}

# Geometry of every node.
#
# For a leaf:
#     x = leaf position
#     y = 0
#
# For a newly created cluster:
#     x = midpoint of its two children
#     y = semantic distance of the merge
node_x = {}
node_y = {}

for i, concept in enumerate(concepts):
    node_x[i] = leaf_x[concept]
    node_y[i] = 0.0


# One geometry record per merge, in EXACT Z order.
merge_geometry = []

for i, row in enumerate(Z):

    left_id = int(row[0])
    right_id = int(row[1])
    distance = float(row[2])

    new_id = n_concepts + i

    cluster_members[new_id] = (
        cluster_members[left_id]
        + cluster_members[right_id]
    )

    left_x = node_x[left_id]
    right_x = node_x[right_id]

    left_y = node_y[left_id]
    right_y = node_y[right_id]

    new_x = (left_x + right_x) / 2

    node_x[new_id] = new_x
    node_y[new_id] = distance

    merge_geometry.append(
        {
            "left_id": left_id,
            "right_id": right_id,
            "new_id": new_id,
            "distance": distance,
            "x": [
                left_x,
                left_x,
                right_x,
                right_x,
            ],
            "y": [
                left_y,
                distance,
                distance,
                right_y,
            ],
        }
    )


# ------------------------------------------------------------
# 3 — Human-readable descriptions
# ------------------------------------------------------------

def node_is_concept(node_id):
    return node_id < n_concepts


def describe_node(node_id):
    """
    Human-readable description of a concept or group.

    Small groups are shown completely:
        {Dog, Wolf}

    Larger groups are abbreviated:
        {Alligator ... Mango}
    """

    members = cluster_members[node_id]

    if len(members) == 1:
        return members[0]

    if len(members) <= 3:
        return "{" + ", ".join(members) + "}"

    return f"{{{members[0]} ... {members[-1]}}}"


def describe_merge(left_id, right_id):
    """Return a human-readable type for one merge."""

    left_is_concept = node_is_concept(left_id)
    right_is_concept = node_is_concept(right_id)

    if left_is_concept and right_is_concept:
        return "Two concepts join"

    if left_is_concept or right_is_concept:
        return "A concept joins a group"

    return "Two groups join"


# ------------------------------------------------------------
# 4 — State
# ------------------------------------------------------------

state = {
    "step": 0
}


# ------------------------------------------------------------
# 5 — Drawing function
# ------------------------------------------------------------

def draw_progressive_dendrogram():

    print(">>> PROGRESSIVE DENDROGRAM V2 <<<")

    step = state["step"]

    fig, ax = plt.subplots(figsize=(11, 6))

    # --------------------------------------------------------
    # Draw exactly the first N merges from Z
    # --------------------------------------------------------

    for merge in merge_geometry[:step]:

        ax.plot(
            merge["x"],
            merge["y"],
            color="0.45",
            linewidth=1.6,
        )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_xlim(
        0,
        10 * len(leaf_labels),
    )

    max_distance = max(
        merge["distance"]
        for merge in merge_geometry
    )

    ax.set_ylim(
        0,
        max_distance * 1.08,
    )

    ax.set_xticks(
        [leaf_x[label] for label in leaf_labels]
    )

    ax.set_xticklabels(
        leaf_labels,
        rotation=45,
        ha="right",
        fontsize=11,
    )

    # --------------------------------------------------------
    # Human-category colors
    #
    # categories and category_colors are expected to have
    # been created by the preceding Lab cells.
    # --------------------------------------------------------


# --------------------------------------------------------
# Human-category colors
# Optional: if categories are available
# --------------------------------------------------------

if "categories" in globals() and "category_colors" in globals():

    for label in ax.get_xticklabels():

        concept = label.get_text()
        category = categories.get(concept)

        if category and category in category_colors:
            label.set_color(
                category_colors[category]
            )
            label.set_fontweight("bold")


    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        f"Building the hierarchy — {scenario_name}\n"
        f"{model_alias} · Step {step}/{total_steps}",
        pad=12,
    )

    ax.set_ylabel("Semantic distance")

    # --------------------------------------------------------
    # Narrative explanation
    #
    # IMPORTANT:
    # merge_geometry[step - 1] and Z[step - 1]
    # describe the SAME merge.
    # --------------------------------------------------------

    if step == 0:

        message = (
            "Start with isolated concepts.\n"
            "Press Next to reveal the first merge."
        )

    else:

        merge = merge_geometry[step - 1]

        left_id = merge["left_id"]
        right_id = merge["right_id"]
        distance = merge["distance"]

        merge_type = describe_merge(
            left_id,
            right_id,
        )

        left_text = describe_node(left_id)
        right_text = describe_node(right_id)

        message = (
            f"Step {step}/{total_steps} — {merge_type}\n"
            f"{left_text}  +  {right_text}\n"
            f"Semantic distance: {distance:.3f}"
        )

    fig.text(
        0.08,
        0.91,
        message,
        ha="left",
        va="top",
        fontsize=11,
        linespacing=1.4,
    )

    fig.subplots_adjust(
        top=0.78,
        bottom=0.22,
    )

    plt.show()


# ------------------------------------------------------------
# 6 — Controls + refresh
# ------------------------------------------------------------

previous_button = widgets.Button(
    description="◀ Previous"
)

next_button = widgets.Button(
    description="Next ▶",
    button_style="primary",
)

reset_button = widgets.Button(
    description="↺ Reset"
)


def refresh():

    clear_output(wait=True)

    # Disable buttons when reaching either end.
    previous_button.disabled = (
        state["step"] == 0
    )

    next_button.disabled = (
        state["step"] == total_steps
    )

    controls = widgets.HBox(
        [
            previous_button,
            next_button,
            reset_button,
        ]
    )

    display(controls)

    draw_progressive_dendrogram()


def previous_step(_):

    if state["step"] > 0:
        state["step"] -= 1

    refresh()


def next_step(_):

    if state["step"] < total_steps:
        state["step"] += 1

    refresh()


def reset_steps(_):

    state["step"] = 0

    refresh()


previous_button.on_click(previous_step)
next_button.on_click(next_step)
reset_button.on_click(reset_steps)


# ------------------------------------------------------------
# 7 — First display
# ------------------------------------------------------------

refresh()

In [ ]:
# ============================================================
# EXPERIMENT — Progressive dendrogram
# V3: Step-by-step construction with synchronized narration
# ============================================================

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.cluster.hierarchy import dendrogram


# ------------------------------------------------------------
# 1 — Final leaf order
#
# scipy is used only to determine the final left-to-right
# ordering of leaves.
# ------------------------------------------------------------

ddata = dendrogram(
    Z,
    labels=concepts,
    no_plot=True,
)

leaf_labels = ddata["ivl"]

# scipy leaf x positions:
# 5, 15, 25, 35, ...
leaf_x = {
    label: 5 + 10 * i
    for i, label in enumerate(leaf_labels)
}


# ------------------------------------------------------------
# 2 — Reconstruct hierarchy directly from Z
#
# Drawing and narration are both driven by the same linkage
# rows, so they remain perfectly synchronized.
# ------------------------------------------------------------

n_concepts = len(concepts)
total_steps = len(Z)

cluster_members = {
    i: [concepts[i]]
    for i in range(n_concepts)
}

node_x = {}
node_y = {}

for i, concept in enumerate(concepts):
    node_x[i] = leaf_x[concept]
    node_y[i] = 0.0


merge_geometry = []

for i, row in enumerate(Z):

    left_id = int(row[0])
    right_id = int(row[1])
    distance = float(row[2])

    new_id = n_concepts + i

    cluster_members[new_id] = (
        cluster_members[left_id]
        + cluster_members[right_id]
    )

    left_x = node_x[left_id]
    right_x = node_x[right_id]

    left_y = node_y[left_id]
    right_y = node_y[right_id]

    new_x = (left_x + right_x) / 2

    node_x[new_id] = new_x
    node_y[new_id] = distance

    merge_geometry.append(
        {
            "left_id": left_id,
            "right_id": right_id,
            "new_id": new_id,
            "distance": distance,
            "x": [
                left_x,
                left_x,
                right_x,
                right_x,
            ],
            "y": [
                left_y,
                distance,
                distance,
                right_y,
            ],
        }
    )


# ------------------------------------------------------------
# 3 — Human-readable descriptions
# ------------------------------------------------------------

def node_is_concept(node_id):
    return node_id < n_concepts


def describe_node(node_id):
    """
    Human-readable description of a concept or group.

    Examples:
        Dog
        {Dog, Wolf}
        {Alligator ... Mango}
    """

    members = cluster_members[node_id]

    if len(members) == 1:
        return members[0]

    if len(members) <= 3:
        return "{" + ", ".join(members) + "}"

    return f"{{{members[0]} ... {members[-1]}}}"


def describe_merge(left_id, right_id):
    """
    Human-readable description of the type of merge.
    """

    left_is_concept = node_is_concept(left_id)
    right_is_concept = node_is_concept(right_id)

    if left_is_concept and right_is_concept:
        return "Two concepts join"

    if left_is_concept or right_is_concept:
        return "A concept joins a group"

    return "Two groups join"


# ------------------------------------------------------------
# 4 — State
# ------------------------------------------------------------

state = {
    "step": 0
}


# ------------------------------------------------------------
# 5 — Drawing function
# ------------------------------------------------------------

def draw_progressive_dendrogram():

    step = state["step"]

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    print(
        f"Building the hierarchy — {scenario_name}"
    )

    print(
        f"{model_alias} · Step {step}/{total_steps}"
    )

    print()

    # --------------------------------------------------------
    # Narrative
    # --------------------------------------------------------

    if step == 0:

        message = (
            "Start with isolated concepts.\n"
            "Press Next to reveal the first merge."
        )

    else:

        merge = merge_geometry[step - 1]

        left_id = merge["left_id"]
        right_id = merge["right_id"]
        distance = merge["distance"]

        merge_type = describe_merge(
            left_id,
            right_id,
        )

        left_text = describe_node(left_id)
        right_text = describe_node(right_id)

        message = (
            f"Step {step}/{total_steps} — {merge_type}\n"
            f"{left_text}  +  {right_text}\n"
            f"Semantic distance: {distance:.3f}"
        )

    print(message)
    print()

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(11, 6)
    )

    # --------------------------------------------------------
    # Draw exactly the first N merges from Z
    # --------------------------------------------------------

    for merge in merge_geometry[:step]:

        ax.plot(
            merge["x"],
            merge["y"],
            color="0.45",
            linewidth=1.6,
        )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_xlim(
        0,
        10 * len(leaf_labels),
    )

    max_distance = max(
        merge["distance"]
        for merge in merge_geometry
    )

    ax.set_ylim(
        0,
        max_distance * 1.08,
    )

    ax.set_xticks(
        [
            leaf_x[label]
            for label in leaf_labels
        ]
    )

    ax.set_xticklabels(
        leaf_labels,
        rotation=45,
        ha="right",
        fontsize=11,
    )

    ax.set_ylabel(
        "Semantic distance"
    )

    # --------------------------------------------------------
    # Optional human-category colors
    # --------------------------------------------------------

    if (
        "categories" in globals()
        and "category_colors" in globals()
    ):

        for label in ax.get_xticklabels():

            concept = label.get_text()
            category = categories.get(concept)

            if (
                category
                and category in category_colors
            ):
                label.set_color(
                    category_colors[category]
                )

                label.set_fontweight(
                    "bold"
                )

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    fig.subplots_adjust(
        bottom=0.25
    )

    plt.show()


# ------------------------------------------------------------
# 6 — Controls
# ------------------------------------------------------------

previous_button = widgets.Button(
    description="◀ Previous"
)

next_button = widgets.Button(
    description="Next ▶",
    button_style="primary",
)

reset_button = widgets.Button(
    description="↺ Reset"
)


# ------------------------------------------------------------
# 7 — Refresh
# ------------------------------------------------------------

def refresh():

    clear_output(
        wait=True
    )

    previous_button.disabled = (
        state["step"] == 0
    )

    next_button.disabled = (
        state["step"] == total_steps
    )

    controls = widgets.HBox(
        [
            previous_button,
            next_button,
            reset_button,
        ]
    )

    display(
        controls
    )

    draw_progressive_dendrogram()


# ------------------------------------------------------------
# 8 — Button callbacks
# ------------------------------------------------------------

def previous_step(_):

    if state["step"] > 0:
        state["step"] -= 1

    refresh()


def next_step(_):

    if state["step"] < total_steps:
        state["step"] += 1

    refresh()


def reset_steps(_):

    state["step"] = 0

    refresh()


previous_button.on_click(
    previous_step
)

next_button.on_click(
    next_step
)

reset_button.on_click(
    reset_steps
)


# ------------------------------------------------------------
# 9 — First display
# ------------------------------------------------------------

refresh()

In [ ]:
# ============================================================
# EXPERIMENT — Progressive dendrogram
# V4: Stable widget layout + synchronized narration
# ============================================================

import html

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.cluster.hierarchy import dendrogram


# ------------------------------------------------------------
# 1 — Final leaf order
# ------------------------------------------------------------

ddata = dendrogram(
    Z,
    labels=concepts,
    no_plot=True,
)

leaf_labels = ddata["ivl"]

leaf_x = {
    label: 5 + 10 * i
    for i, label in enumerate(leaf_labels)
}


# ------------------------------------------------------------
# 2 — Reconstruct hierarchy directly from Z
# ------------------------------------------------------------

n_concepts = len(concepts)
total_steps = len(Z)

cluster_members = {
    i: [concepts[i]]
    for i in range(n_concepts)
}

node_x = {}
node_y = {}

for i, concept in enumerate(concepts):
    node_x[i] = leaf_x[concept]
    node_y[i] = 0.0


merge_geometry = []

for i, row in enumerate(Z):

    left_id = int(row[0])
    right_id = int(row[1])
    distance = float(row[2])

    new_id = n_concepts + i

    cluster_members[new_id] = (
        cluster_members[left_id]
        + cluster_members[right_id]
    )

    left_x = node_x[left_id]
    right_x = node_x[right_id]

    left_y = node_y[left_id]
    right_y = node_y[right_id]

    new_x = (left_x + right_x) / 2

    node_x[new_id] = new_x
    node_y[new_id] = distance

    merge_geometry.append(
        {
            "left_id": left_id,
            "right_id": right_id,
            "new_id": new_id,
            "distance": distance,
            "x": [
                left_x,
                left_x,
                right_x,
                right_x,
            ],
            "y": [
                left_y,
                distance,
                distance,
                right_y,
            ],
        }
    )


# ------------------------------------------------------------
# 3 — Human-readable descriptions
# ------------------------------------------------------------

def node_is_concept(node_id):
    return node_id < n_concepts


def describe_node(node_id):
    """
    Human-readable description of a concept or group.

    Examples:
        Dog
        {Dog, Wolf}
        {Alligator ... Mango}
    """

    members = cluster_members[node_id]

    if len(members) == 1:
        return members[0]

    if len(members) <= 3:
        return "{" + ", ".join(members) + "}"

    return f"{{{members[0]} ... {members[-1]}}}"


def describe_merge(left_id, right_id):

    left_is_concept = node_is_concept(left_id)
    right_is_concept = node_is_concept(right_id)

    if left_is_concept and right_is_concept:
        return "Two concepts join"

    if left_is_concept or right_is_concept:
        return "A concept joins a group"

    return "Two groups join"


# ------------------------------------------------------------
# 4 — State
# ------------------------------------------------------------

state = {
    "step": 0
}


# ------------------------------------------------------------
# 5 — Widgets
# ------------------------------------------------------------

title_html = widgets.HTML()

message_html = widgets.HTML()

previous_button = widgets.Button(
    description="◀ Previous",
    layout=widgets.Layout(width="180px"),
)

next_button = widgets.Button(
    description="Next ▶",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

reset_button = widgets.Button(
    description="↺ Reset",
    layout=widgets.Layout(width="180px"),
)

controls = widgets.HBox(
    [
        previous_button,
        next_button,
        reset_button,
    ],
    layout=widgets.Layout(
        justify_content="center",
        gap="10px",
    ),
)

plot_output = widgets.Output()


# ------------------------------------------------------------
# 6 — Update narrative
# ------------------------------------------------------------

def update_narrative():

    step = state["step"]

    title_html.value = (
        "<div style='margin: 4px 0 2px 0;'>"
        f"<b>Building the hierarchy — "
        f"{html.escape(str(scenario_name))}</b>"
        "&nbsp;&nbsp;·&nbsp;&nbsp;"
        f"{html.escape(str(model_alias))}"
        "&nbsp;&nbsp;·&nbsp;&nbsp;"
        f"Step {step}/{total_steps}"
        "</div>"
    )

    if step == 0:

        message_html.value = (
            "<div style='margin: 2px 0 8px 0;'>"
            "Start with isolated concepts. "
            "Press <b>Next</b> to reveal the first merge."
            "</div>"
        )

        return

    merge = merge_geometry[step - 1]

    left_id = merge["left_id"]
    right_id = merge["right_id"]
    distance = merge["distance"]

    merge_type = describe_merge(
        left_id,
        right_id,
    )

    left_text = html.escape(
        describe_node(left_id)
    )

    right_text = html.escape(
        describe_node(right_id)
    )

    message_html.value = (
        "<div style='margin: 2px 0 8px 0;'>"
        f"<b>{merge_type}</b>"
        "&nbsp;&nbsp;—&nbsp;&nbsp;"
        f"{left_text}"
        "&nbsp;&nbsp;+&nbsp;&nbsp;"
        f"{right_text}"
        "&nbsp;&nbsp;·&nbsp;&nbsp;"
        f"Semantic distance: <b>{distance:.3f}</b>"
        "</div>"
    )


# ------------------------------------------------------------
# 7 — Draw dendrogram
# ------------------------------------------------------------

def draw_progressive_dendrogram():

    step = state["step"]

    with plot_output:

        clear_output(wait=True)

        fig, ax = plt.subplots(
            figsize=(11, 6)
        )

        # ----------------------------------------------------
        # Draw exactly the first N merges from Z
        # ----------------------------------------------------

        for merge in merge_geometry[:step]:

            ax.plot(
                merge["x"],
                merge["y"],
                color="0.45",
                linewidth=1.6,
            )

        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------

        ax.set_xlim(
            0,
            10 * len(leaf_labels),
        )

        max_distance = max(
            merge["distance"]
            for merge in merge_geometry
        )

        ax.set_ylim(
            0,
            max_distance * 1.08,
        )

        ax.set_xticks(
            [
                leaf_x[label]
                for label in leaf_labels
            ]
        )

        ax.set_xticklabels(
            leaf_labels,
            rotation=45,
            ha="right",
            fontsize=11,
        )

        ax.set_ylabel(
            "Semantic distance"
        )

        # ----------------------------------------------------
        # Optional human-category colors
        # ----------------------------------------------------

        if (
            "categories" in globals()
            and "category_colors" in globals()
        ):

            for label in ax.get_xticklabels():

                concept = label.get_text()
                category = categories.get(concept)

                if (
                    category
                    and category in category_colors
                ):
                    label.set_color(
                        category_colors[category]
                    )

                    label.set_fontweight(
                        "bold"
                    )

        fig.subplots_adjust(
            bottom=0.25
        )

        plt.show()

        plt.close(fig)


# ------------------------------------------------------------
# 8 — Refresh
# ------------------------------------------------------------

def refresh():

    previous_button.disabled = (
        state["step"] == 0
    )

    next_button.disabled = (
        state["step"] == total_steps
    )

    update_narrative()
    draw_progressive_dendrogram()


# ------------------------------------------------------------
# 9 — Button callbacks
# ------------------------------------------------------------

def previous_step(_):

    if state["step"] > 0:
        state["step"] -= 1

    refresh()


def next_step(_):

    if state["step"] < total_steps:
        state["step"] += 1

    refresh()


def reset_steps(_):

    state["step"] = 0

    refresh()


previous_button.on_click(
    previous_step
)

next_button.on_click(
    next_step
)

reset_button.on_click(
    reset_steps
)


# ------------------------------------------------------------
# 10 — Interface
# ------------------------------------------------------------

interface = widgets.VBox(
    [
        title_html,
        message_html,
        controls,
        plot_output,
    ]
)

display(interface)

refresh()